# Nghe — build the audio

Run the cells top to bottom. Everything happens on Kaggle's machines, not yours.

This builds **two accents** — Southern and Northern — as separate folders of clips, `audio/south/` and `audio/north/`. The app lets you choose which accent to practise with.

Generation runs **inside this notebook** using one loaded copy of the model, so it shows live progress and there is nothing to hang.

## 1. Turn on GPU, internet, and your token

All three are in the **right-hand sidebar** (open it with the **`<`** arrow, then **Session options**):

- **Accelerator → GPU T4 x2** (or **GPU P100**). We use one GPU.
- **Internet → On.** Needed to install VieNeu, clone the repo and push back. Kaggle requires a phone-verified account.
- **Add-ons → Secrets →** add a secret named `GITHUB_TOKEN` and tick it to attach it to this notebook.

The token is a GitHub fine-grained token with **Contents: Read and write** on `councilgritter/nghe`. After a session restart the secret may need re-ticking. (The repo is public, so the clone works without it — the token is only needed for the push at the end.)

## 2. Get the repo

Clones the repo into the working folder and moves into it. The token is never printed. Re-running just pulls the latest.

In [ ]:
import os, subprocess

REPO  = 'councilgritter/nghe'
EMAIL = 'sam.saunders96@gmail.com'
NAME  = 'councilgritter'

def get_token():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret('GITHUB_TOKEN')
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get('GITHUB_TOKEN')
    except Exception:
        return os.environ.get('GITHUB_TOKEN')

TOKEN = get_token()
WORK = '/kaggle/working' if os.path.isdir('/kaggle/working') else '/content'
REPO_DIR = os.path.join(WORK, 'nghe')
auth = f'https://{TOKEN}@github.com/{REPO}.git' if TOKEN else f'https://github.com/{REPO}.git'

if os.path.isdir(os.path.join(REPO_DIR, '.git')):
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--rebase'], check=False)
else:
    subprocess.run(['git', 'clone', auth, REPO_DIR], check=True)
subprocess.run(['git', '-C', REPO_DIR, 'config', 'user.email', EMAIL], check=True)
subprocess.run(['git', '-C', REPO_DIR, 'config', 'user.name', NAME], check=True)
if TOKEN:
    subprocess.run(['git', '-C', REPO_DIR, 'remote', 'set-url', 'origin', auth], check=True)
os.chdir(REPO_DIR)
print('working in', REPO_DIR, '| token:', 'yes' if TOKEN else 'no (clone only)')

## 3. Install VieNeu

VieNeu is the Vietnamese text-to-speech model; librosa and soundfile trim and normalise the clips. A couple of minutes the first time. ffmpeg is already on Kaggle's image; the check just confirms it.

In [ ]:
!pip install -q vieneu librosa soundfile
!ffmpeg -version | head -1

## 4. Load the model and see the voices

Loads VieNeu **once** (`tts`) and prints the voices with their accent. Everything below reuses this one model.

The description is **Gender · Region · Style**: `Nam`/`Nữ` = male/female, and `Bắc`/`Trung`/`Nam` = North/Central/South. The chosen voices are **Adam** (Southern male) and **Ngọc Huyền** (Northern female); swap either in the next step.

In [ ]:
import importlib, generate_clips as gc
importlib.reload(gc)          # pick up any pulled changes
tts = gc.load_tts()
for desc, name in gc.list_voices(tts):
    print(f'{name:24s} {desc}')

## 5. Audition

Set the voice and the two speeds per accent, then run the cell after it to hear 40 of the commonest syllables in each — made by the real pipeline, so this is exactly what the full run produces. Change something, put the region in `RESET`, and re-run. Nothing is committed here.

- **VOICES** — a voice name from the list above.
- **SPEED** — tempo of the level tones (ngang, huyền, sắc, nặng).
- **DIP** — tempo of the contour tones hỏi and ngã, a little slower so the ear can track their shape.
- **RESET** — regions to wipe and remake, e.g. `['north']`.

Each clip is one syllable said alone — trimmed, loudness-matched, padded with 0.5s of silence, and slowed. The app's *Slower* button and replay sit on top of this.

In [ ]:
VOICES = {'south': 'Adam',      'north': 'Ngọc Huyền'}
SPEED  = {'south': 0.9,         'north': 0.9}     # level tones
DIP    = {'south': 0.8,         'north': 0.8}     # hỏi and ngã
RESET  = []                                        # e.g. ['north'] to remake a region

In [ ]:
import glob, csv, os, shutil
import IPython.display as ipd

names = {r['clip_id']: r['syllable'] for r in
         csv.DictReader(open('vietnamese_clip_manifest.csv', encoding='utf-8-sig'))}

def build(region, limit=None):
    if region in RESET:
        shutil.rmtree(f'audio/{region}', ignore_errors=True)
        print('wiped audio/' + region)
    gc.run(region, VOICES[region], tts, used_in='data.json',
           speed=SPEED[region], speed_dip=DIP[region], limit=limit)

for region in VOICES:
    build(region, limit=40)
    for f in sorted(glob.glob(f'audio/{region}/*.mp3'))[:10]:
        cid = os.path.basename(f)[:-4]
        print(region, cid, names.get(cid))
        ipd.display(ipd.Audio(f))

RESET = []   # clear so a re-run does not wipe unless you ask again

## 6. The full run

Once the audition sounds right, this makes every clip the app plays (about 6,000 per accent), one accent at a time, with a live progress line. It **skips clips that already exist**, so if the session drops you just re-run it.

Each accent is pushed to GitHub as it finishes, so completed accents are safe. On Kaggle a GPU session runs for hours, so both usually fit in one sitting; edit `TODO` to do one at a time.

In [ ]:
TODO = ['south', 'north']

def git(*args, check=True):
    return subprocess.run(['git', *args], check=check)

def publish(message, *paths):
    git('add', *paths)
    if git('diff', '--cached', '--quiet', check=False).returncode == 0:
        print('nothing new to commit'); return
    git('-c', 'commit.gpgsign=false', 'commit', '-m', message)
    git('pull', '--rebase', check=False)
    git('push', 'origin', 'HEAD:main')

for region in TODO:
    build(region)                       # no limit = the whole accent
    print(len(glob.glob(f'audio/{region}/*.mp3')), 'clips in', region)
    publish(f'Add {region} clips', f'audio/{region}')

## 7. Done

Both accents are on GitHub, and the live site updates within a minute of each push. Open it on your phone and have a native Vietnamese speaker check the clips — that is the real test.

**Removing a bad clip later.** Delete `audio/<accent>/<clip_id>.mp3` and push (with GitHub Desktop, or by re-running here). The app falls back to the browser voice for anything missing.

**Changing a voice or speed.** Edit `VOICES`/`SPEED`/`DIP` in step 5, add that region to `RESET`, and re-run steps 5–6. Step 6 only makes what is missing, so `RESET` is how you force a remake.